In [9]:
import warnings

warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import sklearn as skl
import pycountry
import plotly.express as px
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    KFold,
)
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    brier_score_loss,
    log_loss,
    f1_score,
    roc_auc_score,
    adjusted_rand_score,
    silhouette_score,
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CUR_DIR = os.getcwd()
BOOKS_DATA_PATH = os.path.join(CUR_DIR, "books_data/books.csv")
RATINGS_DATA_PATH = os.path.join(CUR_DIR, "books_data/ratings.csv")
USERS_DATA_PATH = os.path.join(CUR_DIR, "books_data/users.csv")

df_books = pd.read_csv(BOOKS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\").drop(columns = ['Image-URL-S', 'Image-URL-M', 'Image-URL-L'])
df_ratings = pd.read_csv(RATINGS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"')
df_users = pd.read_csv(USERS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\", index_col=0)

print(f"Books data    loaded {len(df_books):,} rows, {df_books.shape[1]} columns.")
print(f"Ratings data  loaded {len(df_ratings):,} rows, {df_ratings.shape[1]} columns.")
print(f"Users data    loaded {len(df_users):,} rows, {df_users.shape[1]} columns.")

df_active_countries = df_users['Location'].str.split(',').str[-1].str.strip()
unique_active_countries = df_active_countries.unique()

Books data    loaded 271,379 rows, 5 columns.
Ratings data  loaded 1,149,780 rows, 3 columns.
Users data    loaded 278,858 rows, 2 columns.


In [10]:
"""
METHOD 1
"""
countries = []
for country in pycountry.countries:
    countries.append(country.name.lower())

mutual = []
for mutual_country in countries:
    if mutual_country in unique_active_countries:
        mutual.append(mutual_country)

#ADD MANUALLY countries not in mutual
mutual += "usa", "russia", "iran", "vietnam", "u.a.e", "turkey", "taiwan", "syria", "venezuela", "south korea", "czech republic"
 
total_users_method1 = 0
for country in mutual:
    total_users_method1 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning;            METHOD 1: {round(total_users_method1/len(df_users)*100, 2)}")

"""
METHOD 2
this is an additional validation step of the mutual list
"""
countries_counts = df_active_countries.value_counts()
counts_list = countries_counts[countries_counts>50].index.tolist()
counts_list.remove("")

total_users_method2 = 0
for country in counts_list:
    total_users_method2 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning; only using METHOD 2: {round(total_users_method2/len(df_users)*100, 2)}")
print(f"Countries, having more than 50 customers, excluded from the countries list are: {set(counts_list)- set(mutual)}")

df_users['Country'] = df_users['Location'].str.split(',').str[-1].str.strip().str.lower()
df_users['Country'] = df_users['Country'].apply(lambda x: x if x in mutual else "unknown")

Percentage of users kept after cleaning;            METHOD 1: 97.92
Percentage of users kept after cleaning; only using METHOD 2: 97.38
Countries, having more than 50 customers, excluded from the countries list are: {'yugoslavia', 'españa'}


In [11]:
#merge all data into one table - df_main
df_main = pd.merge(df_ratings,df_users, how='left', on='User-ID')
df_main = pd.merge(df_main, df_books, how= 'left', on='ISBN')
df_main = df_main.drop(['Location','Country'], axis = 1)
#restricts data only to top publishers
top_publishers = df_main['Publisher'].value_counts().head(10).index.to_list()
df_main = df_main[df_main['Publisher'].isin(top_publishers)]

#creates book-read-count column
read_count = df_main['Book-Title'].value_counts()
read_count.name = 'book-read-count'

#creates author-read-count column
author_counts = df_main['Book-Author'].value_counts()
author_counts.name = 'author-read-count'

#adds author_counts and read_count to df_main
df_main = pd.merge(df_main, author_counts, how='left', on='Book-Author')
df_main = pd.merge(df_main, read_count, how='left', on='Book-Title')

#create df_book_list table for recommendation system
df_book_list = (
    df_main[
        [
            'ISBN',
            'Book-Title',
            'Book-Author',
            'Year-Of-Publication',
            'Publisher',
            'book-read-count',
            'author-read-count'
        ]
    ]
    .drop_duplicates('ISBN')
    .copy()
)

## Book ISBN search based on title for future actions


In [12]:
def search_books_by_title(search_text, n=10):
    """
    Search books by title so you can find the correct ISBN.
    """
    search_text = str(search_text).lower()

    results = df_book_list[
        df_book_list['Book-Title']
        .astype(str)
        .str.lower()
        .str.contains(search_text, na=False)
    ].copy()

    results = results.sort_values(
        by='book-read-count',
        ascending=False
    )

    return results.head(n)

## Co-reader vote recommender

## Recommendation model logic

This recommender uses a co-reader voting approach. The input is a list of seed books represented by ISBNs. The model first finds users who are connected to these seed books, then recommends other books read by those users.

The function has two cases controlled by `k`:

- If `k = 0`, the model uses normal co-reader voting. A user counts as a match if they read or interacted with at least one input book.
- If `k != 0`, the model uses liked-seed voting. A user only counts as a match if they rated at least one input book with a rating greater than or equal to `k`.

For each matched user, the model counts how many input books they matched. This count becomes their vote weight. Other books read by that user receive votes equal to this weight. For example, if a user matched 2 input books, every other book they read receives 2 votes.

The model then calculates:

- `coreader_vote_score` or `liked_seed_vote_score`: total weighted votes a candidate book received.
- `matched_reader_count`: number of matched users who read the candidate book.
- `global_reader_count`: number of users in the whole cleaned dataset who read the candidate book.

The final recommendations are the books with the highest vote scores, excluding the original input books.

In [13]:
def recommender(seed_isbns, k=0):
    # Clean input ISBNs
    seed_isbns = [str(isbn).strip() for isbn in seed_isbns]
    seed_isbns = list(set(seed_isbns))

    # Read/interacted events from df_main
    read_events = df_main[["User-ID", "ISBN"]].drop_duplicates()

    # Keep only seed books that exist in df_main
    valid_seed_isbns = [
        isbn for isbn in seed_isbns
        if isbn in set(df_main["ISBN"])
    ]

    print("Input seed ISBNs:", seed_isbns)
    print("Valid seed ISBNs:", valid_seed_isbns)

    if len(valid_seed_isbns) == 0:
        raise ValueError("None of the seed ISBNs exist in df_main.")



    # Case 1: k = 0
    # users count if they read/interacted with seed books
    if k == 0:
        print("Mode: co-reader votes based on read/interacted seed books")

        seed_reads = read_events[ read_events["ISBN"].isin(valid_seed_isbns)].copy()

        if seed_reads.empty:
            raise ValueError("No users read the selected seed books.")

        user_seed_matches = (seed_reads.groupby("User-ID")["ISBN"].nunique())
        user_seed_matches.name = "seed_match_count"
        user_seed_matches = user_seed_matches.reset_index()

        score_column = "seed_match_count"
        final_score_name = "coreader_vote_score"

    # Case 2: k != 0
    # Liked-seed logic:
    # users count only if they rated seed books >= k
    else:
        print("Mode: liked-seed votes based on seed ratings >= k")
        print("rating threshold:", k)

        liked_seed_ratings = df_main[(df_main["ISBN"].isin(valid_seed_isbns)) & (df_main["Book-Rating"] >= k)].copy()

        if liked_seed_ratings.empty:
            raise ValueError("No users rated the seed books above or equal to k.")

        user_seed_matches = (liked_seed_ratings.groupby("User-ID")["ISBN"].nunique())
        user_seed_matches.name = "liked_seed_count"
        user_seed_matches = user_seed_matches.reset_index()

        score_column = "liked_seed_count"
        final_score_name = "liked_seed_vote_score"
    print("Matched users:", len(user_seed_matches))



    # Get all other books read by matched users
    candidate_reads = pd.merge(read_events, user_seed_matches, how="inner", on="User-ID")

    # Do not recommend the input seed books
    candidate_reads = candidate_reads[~candidate_reads["ISBN"].isin(valid_seed_isbns)].copy()
    if candidate_reads.empty:
        return pd.DataFrame()

    # Score candidate books
    scores = (
        candidate_reads
        .groupby("ISBN")
        .agg(
            vote_score=(score_column, "sum"),
            matched_reader_count=("User-ID", "nunique")
        ).reset_index())
    scores = scores.rename(columns={"vote_score": final_score_name})

    # Global reader count
    global_reader_counts = (read_events.groupby("ISBN")["User-ID"].nunique())

    global_reader_counts.name = "global_reader_count"
    global_reader_counts = global_reader_counts.reset_index()

    scores = pd.merge(scores, global_reader_counts, how="left", on="ISBN")
    scores = scores.sort_values(by=[final_score_name, "matched_reader_count"], ascending=False)

    recommendations = pd.merge(scores, df_book_list, how="left", on="ISBN")

    return recommendations

## Recomender Test

In [14]:
seed_isbns = [
    "0451139712",   # The Stand
    "0451157443",   # Carrie
    "0743424425	",  # The Shining
    "0451184963",   # Insomnia
    "044021145X",   # The Firm
    "0345337662",   # Interview with the Vampire
    "0440224675",   # Hannibal
    "0399501487",   # Lord of the Flies
    "0671027360",   # Angels and Demons

]

In [15]:
recommender(seed_isbns, k=0)

Input seed ISBNs: ['0440224675', '0743424425', '0451184963', '0671027360', '0451139712', '0345337662', '0399501487', '0451157443', '044021145X']
Valid seed ISBNs: ['0743424425', '0451184963', '0345337662', '0451157443']
Mode: co-reader votes based on read/interacted seed books
Matched users: 720


,ISBN,coreader_vote_score,matched_reader_count,global_reader_count,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,0345313860,194,172,301,"The Vampire Lestat (Vampire Chronicles, Book II)",ANNE RICE,1986.0,Ballantine Books,301,656
1,0345351525,162,141,273,The Queen of the Damned (Vampire Chronicles (P...,Anne Rice,1993.0,Ballantine Books,273,2219
2,0345370775,136,111,466,Jurassic Park,Michael Crichton,1999.0,Ballantine Books,468,1914
3,034538475X,113,99,193,The Tale of the Body Thief (Vampire Chronicles...,Anne Rice,1993.0,Ballantine Books,193,2219
4,0446672211,102,86,585,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts,1998.0,Warner Books,585,819
...,...,...,...,...,...,...,...,...,...,...
18747,2266096400,1,1,1,Le CrÃ?Â©puscule des elfes,Jean-Louis Fetjaine,2002.0,Pocket,1,1
18748,2266102931,1,1,1,Fritna,GisÃ?Â¨le Halimi,2001.0,Pocket,1,1
18749,2266105701,1,1,1,Les mauvaises pensÃ?Â©es,Laurent Seksik,2001.0,Pocket,1,1
18750,2266107534,1,1,3,La citÃ?Â© de la joie,Dominique Lapierre,2000.0,Pocket,3,26


In [16]:
recommender(seed_isbns,k=8)

Input seed ISBNs: ['0440224675', '0743424425', '0451184963', '0671027360', '0451139712', '0345337662', '0399501487', '0451157443', '044021145X']
Valid seed ISBNs: ['0743424425', '0451184963', '0345337662', '0451157443']
Mode: liked-seed votes based on seed ratings >= k
rating threshold: 8
Matched users: 218


,ISBN,liked_seed_vote_score,matched_reader_count,global_reader_count,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,0345313860,60,59,301,"The Vampire Lestat (Vampire Chronicles, Book II)",ANNE RICE,1986.0,Ballantine Books,301,656
1,0345351525,48,47,273,The Queen of the Damned (Vampire Chronicles (P...,Anne Rice,1993.0,Ballantine Books,273,2219
2,034538475X,40,39,193,The Tale of the Body Thief (Vampire Chronicles...,Anne Rice,1993.0,Ballantine Books,193,2219
3,0345370775,29,28,466,Jurassic Park,Michael Crichton,1999.0,Ballantine Books,468,1914
4,0451156609,29,25,170,The Tommyknockers,Stephen King,1994.0,Signet Book,170,5667
...,...,...,...,...,...,...,...,...,...,...
6675,2266118536,1,1,3,Myrtille Ã?Â la plage,Olivier Mau,2003.0,Pocket,3,4
6676,2266120166,1,1,11,La Rage au coeur,Ingrid Betancourt,2002.0,Pocket,11,11
6677,2266130250,1,1,2,"Rupture dans le rÃ?Â©el, tome 1-1 : GÃ?Â©nÃ?Â©se",Peter F. Hamilton,2003.0,Pocket,2,24
6678,2266131516,1,1,4,La MÃ?Â©thode simple pour en finir avec la cig...,Allen Carr,2003.0,Pocket,4,4


## Extra Data cleaning for Regression

In [17]:
#creates book-rate-count column
df_main = df_main[df_main['Book-Rating'] != 0]
rate_count = df_main['Book-Title'].value_counts()
rate_count.name = 'book-rate-count'

#adds rate_count to df_main
df_main = pd.merge(df_main, rate_count, how= 'left', on='Book-Title')

median_age = df_main['Age'].median()
df_main['Age'] = df_main['Age'].fillna(median_age)
K = 8
df_main['rating-over-k'] = df_main['Book-Rating'].apply(lambda row: 1 if row >= K else 0)

display(df_main)

NUM_COLS = [
    'book-rate-count',
    'book-read-count',
    'author-read-count',
    'Year-Of-Publication',
    'Age',
]
CAT_COLS = [
    'Publisher',
]
TARGET = [
    'rating-over-k'
]

,User-ID,ISBN,Book-Rating,Age,Book-Title,Book-Author,Year-Of-Publication,Publisher,author-read-count,book-read-count,book-rate-count,rating-over-k
0,276747,0671537458,9,25.0,Waiting to Exhale,Terry McMillan,1995.0,Pocket,419,88,17,1
1,276755,0451166892,5,32.0,The Pillars of the Earth,Ken Follett,1996.0,Signet Book,821,228,87,0
2,276762,0380711524,5,25.0,See Jane Run,Joy Fielding,1992.0,Avon,182,46,16,0
3,276772,0553572369,7,35.0,Pay Dirt (Mrs. Murphy Mysteries (Paperback)),RITA MAE BROWN,1996.0,Bantam,383,26,8,0
4,276788,0345443683,8,35.0,"Blackwood Farm (Rice, Anne, Vampire Chronicles.)",ANNE RICE,2003.0,Ballantine Books,656,49,20,1
...,...,...,...,...,...,...,...,...,...,...,...,...
80769,276688,0553575090,7,35.0,Deception on His Mind,ELIZABETH GEORGE,1998.0,Bantam,279,89,25,0
80770,276688,0553575104,6,35.0,In Pursuit of the Proper Sinner,Elizabeth George,2000.0,Bantam Books,512,79,20,0
80771,276688,0671015591,2,35.0,Tiger's Palette (Caroline Canfield Mysteries),Jacqueline Fiedler,1998.0,Pocket,5,3,2,0
80772,276688,0671563149,6,35.0,MUDDY WATER (Peter Bartholomew Mysteries),Sally Gunning,1997.0,Pocket,30,3,1,0


In [15]:
def train_best_classifier(X_train, y_train, seed):
    """Return a fitted classifier with predict_proba ready for held-out evaluation."""
    # YOUR CODE HERE
    from sklearn.model_selection import GridSearchCV
    from sklearn.tree import DecisionTreeClassifier

    def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
        numeric_step = StandardScaler() if scale_numeric else "passthrough"
        return ColumnTransformer(
            [
                ("num", numeric_step, NUM_COLS),
                (
                    "cat",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                    CAT_COLS,
                ),
            ]
        )

    def make_pipeline_for(model_name: str, **params) -> Pipeline:
        if model_name == "logistic":
            estimator = LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                **params,
            )
            scale_numeric = True
        elif model_name == "tree":
            estimator = DecisionTreeClassifier(
                random_state=RANDOM_STATE,
                **params,
            )
            scale_numeric = False
        elif model_name == "random_forest":
            estimator = RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=1,
                **params,
            )
            scale_numeric = False
        elif model_name == "gradient_boosting":
            estimator = GradientBoostingClassifier(
                random_state=RANDOM_STATE,
                **params,
            )
            scale_numeric = False
        else:
            raise ValueError(f"Unknown model_name: {model_name}")

        return Pipeline(
            [
                ("pre", make_preprocessor(scale_numeric=scale_numeric)),
                ("model", estimator),
            ]
        )


    def cv_f1_scores(label: str, model: Pipeline, X_data, y_data, cv) -> pd.DataFrame:
        # Return one row per fold for positive-class F1.
        scores = cross_val_score(model, X_data, y_data, cv=cv, scoring="f1")
        return pd.DataFrame(
            {"model": label, "fold": np.arange(1, len(scores) + 1), "f1": scores}
        )


    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    rf_grid = GridSearchCV(
        make_pipeline_for("random_forest", n_estimators=100),
        param_grid={
            "model__max_depth": [10, 20, None],
            "model__min_samples_leaf": [1, 5],
        },
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        scoring="f1",
    )
    rf_grid.fit(X_train, y_train)
    rf_best_params = {k.replace("model__", ""): v for k, v in rf_grid.best_params_.items()}

    gb_grid = GridSearchCV(
        make_pipeline_for("gradient_boosting"),
        param_grid={
            "model__n_estimators": [60, 100],
            "model__learning_rate": [0.05, 0.10],
            "model__max_depth": [2, 3],
        },
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        scoring="f1",
    )
    gb_grid.fit(X_train, y_train)
    gb_best_params = {k.replace("model__", ""): v for k, v in gb_grid.best_params_.items()}

    candidate_models = {
        "logistic": make_pipeline_for("logistic"),
        #"shallow tree": make_pipeline_for("tree", max_depth=4, min_samples_leaf=100),
        #"deep tree": make_pipeline_for("tree", max_depth=None, min_samples_leaf=1),
        "random forest": make_pipeline_for(
            "random_forest", n_estimators=100, **rf_best_params
        ),
        "gradient boosting": make_pipeline_for(
            "gradient_boosting",
            **gb_best_params,
        ),
    }

    fold_tables = [
        cv_f1_scores(label, model, X_train, y_train, cv5)
        for label, model in candidate_models.items()
    ]
    fold_scorecard = pd.concat(fold_tables, ignore_index=True)
    summary_scorecard = (
        fold_scorecard.groupby("model")["f1"]
        .agg(mean_f1="mean", std_f1=lambda s: s.std(ddof=0))
        .sort_values("mean_f1", ascending=False)
    )

    display(fold_scorecard.pivot(index="fold", columns="model", values="f1").round(3))
    display(summary_scorecard.round(3))

In [16]:
df_subset = df_main.groupby(TARGET, group_keys=False).apply(lambda x: x.sample(frac=0.10))
display(df_subset)
x = df_subset[NUM_COLS+CAT_COLS]
y = df_subset[TARGET]

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size= 0.2, random_state=RANDOM_STATE)
model = train_best_classifier(X_train,y_train, RANDOM_STATE)

,User-ID,ISBN,Book-Rating,Age,Book-Title,Book-Author,Year-Of-Publication,Publisher,author-read-count,book-read-count,book-rate-count,rating-over-k
57244,193746,0380802252,5,35.0,Baroque and Desperate (Den of Antiquity),Tamar Myers,1999.0,Avon,415,14,4,0
78216,267033,044651862X,7,35.0,The Celestine Prophecy (Celestine Prophecy),James Redfield,1994.0,Warner Books,399,188,74,0
11322,35445,0671749838,6,31.0,ALWAYS KISS WITH YOUR WHISKERS: LOVE ADVICE FR...,Liz Nickles,1991.0,Pocket,17,5,4,0
70576,241565,0671824813,5,64.0,GOODBYE JANETTE,Robbins,1982.0,Pocket,68,14,2,0
71227,242718,0553569058,5,35.0,The Robber Bride,Margaret Atwood,1995.0,Bantam,162,99,23,0
...,...,...,...,...,...,...,...,...,...,...,...,...
54271,182527,0553258915,9,62.0,The Two Mrs. Grenvilles,Dominick Dunne,1986.0,Bantam Books,181,42,10,1
18261,60419,0553347179,8,36.0,Entropy: Into the Greenhouse World (New Age Book),Jeremy Rifkin,1990.0,Bantam Books,3,2,1,1
30018,99204,0425169863,8,35.0,Point of Origin,Patricia Daniels Cornwell,1999.0,Berkley Publishing Group,1602,188,69,1
40300,135149,0140084428,8,35.0,English Creek (Contemporary American Fiction),Ivan Doig,1990.0,Penguin Books,17,8,2,1


model,gradient boosting,logistic,random forest
fold,,,
1,0.725,0.719,0.719
2,0.724,0.720,0.711
3,0.723,0.711,0.709
4,0.714,0.715,0.708
5,0.730,0.714,0.707


,mean_f1,std_f1
model,,
gradient boosting,0.723,0.005
logistic,0.716,0.003
random forest,0.711,0.004
